# TÜV NORD source adapter

Control panel for `tuev_nord.run` (`1_scrapers/tuev_nord/src/run.py`). All logic lives in the module; this notebook only calls it or reads its run folders. Every path comes from `core.paths`. tuev-nord.de needs no key and no login; plain HTTP works (no Cloudflare, 2026-10-06).

**What the source is.** One TYPO3 page, `www.tuev-nord.de/de/stationen/`. All locations are in the attribute `data-locations` (HTML-escaped JSON) of the map div, plus the text "<N> Stationen gefunden". Fields per record: `id, name, street, plz, city, latitude, longitude, serviceType, link, isOffice`. `serviceType` is `HU`, `HU-UNG` or `UNG`; **HU site = `HU` or `HU-UNG`**. Offices (`isOffice`) are appraisal-only (`UNG`) and have no station page; their link carries a volatile token and is never used as an id. The unique key is **id + normalised address** (TÜV NORD reuses an id for several offices).

**What a full run does.** Stations page, A-Z list `/de/stationen/liste-aller-stationen/` (cross-check by page slug), 3 detail pages (one own, one partner, one HU-only) and up to 8 pages that explain stations which are in only one of map and list. About 12 requests, under a minute, 2-4 s between requests, fresh session without cookies. **PASS** needs every `hard` check: counter equals parsed records, all three `serviceType` values present (page not filtered), no other value, all ten fields on every record, map and list agree within 5 %, HU count not down more than 10 % against the last passing full run. Findings that need a human (unclassified names, reused ids, shared addresses, stations without a page, list-only pages) are `review` and never stop a run. A PASS shows consistency between two sources, **not completeness**: a station in neither stays invisible.

| Mode | Requests | Goes to |
|---|---|---|
| `simulated` | none (fake site with known truth, must PASS) | `test_output/` |
| `full` | live, ~12, needs `confirm_full_run=True` | `output/` |
| `rebuild` | none, re-parses the raw pages of an earlier `full` run | `output/` |

A run folder holds `tuev_nord_all_<date>.csv`, `tuev_nord_hu_<date>.csv`, `tuev_nord_run_report_<date>.md` (the full report), `run_summary.json`, `query_log.csv` and `raw/` (the HTML pages as received; a few MB, only in `full` runs). Files contain operator names and stay out of git.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import pandas as pd

from core import paths
from tuev_nord import run as nord

ROOT = paths.project_root()
OUT = paths.output_dir("tuev_nord", "full")  # official runs (full, rebuild)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

def official_runs():
    return sorted(p for p in OUT.iterdir() if (p / paths.RUN_SUMMARY_JSON).exists()) if OUT.is_dir() else []

print("Runs in output/:", [p.name for p in official_runs()])
ROOT

## 2. Inspect a run (no requests)

Reads a run folder from disk, so it works right after a kernel restart. By default the newest official run; to look at another one, set `RUN` to its folder name.

In [ ]:
runs = official_runs()
RUN = runs[-1] if runs else None
# RUN = OUT / "2026-10-06_1116_rebuild"

S = json.loads((RUN / paths.RUN_SUMMARY_JSON).read_text(encoding="utf-8"))
ALL = pd.read_csv(next(RUN.glob("tuev_nord_all_*.csv")), sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
HU = pd.read_csv(next(RUN.glob("tuev_nord_hu_*.csv")), sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
for col in ("is_hu_site", "is_office", "needs_review", "psp_suspect"):
    ALL[col] = ALL[col] == "True"
    HU[col] = HU[col] == "True"

print(RUN.relative_to(ROOT))
print(S["status"], "-", S["outcome"], "| mode", S["mode"], "| requests", S.get("requests_made"))
print("Counts:", S["counts"])
print("Drop check:", S["drop_check"])
print(f"{len(ALL)} records, {len(HU)} HU sites")

In [ ]:
# Checks: failures first (hard = stops the run, review = needs a human)
chk = pd.DataFrame(S["checks"])
chk["ok"] = chk["ok"].map({True: "yes", False: "NO"})
chk.sort_values("ok", kind="stable")[["severity", "ok", "check", "detail"]]

In [ ]:
# Counts: serviceType, site_type and isOffice
print("serviceType:", S["breakdowns"]["service_type"], "| isOffice:", S["breakdowns"]["is_office"])
print()
pd.crosstab(ALL["site_type"], ALL["service_type"], margins=True)

In [ ]:
# HU sites by first PLZ digit and site_type (footprint)
pd.crosstab(HU["site_type"], HU["zip"].str[:1], margins=True)

### Cross-check with the A-Z list

Stations of the map data against the list, by page slug. Offices have no page and are not in the list.

In [ ]:
cr = S["cross"]
print({k: cr[k] for k in ("map_stations", "list_items", "both")}, "| only in map:", len(cr["only_in_map"]), "| only in list:", len(cr["only_in_list"]))
print("\nOnly in the map data:")
display(pd.DataFrame(cr["only_in_map"]))
print("Only in the list:")
display(pd.DataFrame(cr["only_in_list"]))
print("Pages fetched to explain them (page_exists = False means the server answered with the finder page):")
pd.DataFrame(S["extra_pages"])[["slug", "page_exists", "lists_hu_service", "address_lines", "map_coords", "booking_station_ids", "note"]]

### Findings that need a human

In [ ]:
f = S["findings"]
print("Unclassified site_type (never guessed):")
display(pd.DataFrame(f["unclassified"]))
print("Ids used for several addresses:")
display(pd.DataFrame([x for g in f["ids_reused_with_other_address"] for x in g["locations"]]))
print("Addresses listed under several ids:")
pd.DataFrame([{"address": g["address"], **x} for g in f["addresses_shared_by_several_ids"] for x in g["locations"]])

In [ ]:
review = ALL[ALL["needs_review"]]
print(f"{len(review)} rows are flagged for review (they stay in the export); flags:")
print(pd.Series([x for v in ALL["dq_flags"] for x in v.split("; ") if x]).value_counts().to_string())
review[["location_id", "name_clean", "zip", "city", "service_type", "site_type", "dq_flags"]]

In [ ]:
# Operators with several HU sites (name-based, not merged)
ops = HU[HU["operator_name"] != ""].groupby("operator_name").size().sort_values(ascending=False)
ops[ops > 1]

In [ ]:
# The 3 detail pages of the run: what a station page offers
for d in S["detail_pages"]:
    x = d["description"]
    print(f"{d['role']}: {d['url']} (HTTP {d['status']})")
    print("  address:", x.get("address_lines"), "| hours found:", x.get("has_opening_hours"), "| lists HU:", x.get("lists_hu_service"))
    print("  partner note:", x.get("partner_note") or "-", "| booking stationId:", x.get("booking_station_ids"))

### Spot check

The five random HU sites chosen by the run (seeded by the run date, so they are the same on every rebuild), then a fresh draw.

In [ ]:
display(pd.DataFrame(S["spot_check"])[["location_id", "name", "street", "zip", "city", "service_type", "site_type", "detail_url"]])
HU.sample(5, random_state=7)[["location_id", "name_clean", "street", "zip", "city", "service_type", "site_type"]]

## 3. Rebuild (offline, no requests)

Re-parses the raw pages of an earlier `full` run with the current code and writes a `rebuild` run folder in `output/`. Use it after a parser change. Each execution makes a new folder; nothing is overwritten.

In [ ]:
from core import paths
from tuev_nord import run as nord

fulls = [p for p in official_runs() if p.name.endswith("_full") and (p / "raw" / "stationen.html").exists()]
SOURCE = fulls[-1]  # or set a run folder explicitly
rb = nord.run(mode="rebuild", source_run=SOURCE)
print(rb.status, "-", rb.summary["outcome"], "|", rb.summary["counts"])
print(rb.run_dir.relative_to(paths.project_root()))
pd.DataFrame(rb.summary["checks"]).query("not ok")[["severity", "check", "detail"]]

## 4. Fetch

Sections 4a and 4b write one run folder `YYYY-MM-DD_HHMM_<mode>`. The cells are self-contained: they run right after a kernel restart without the cells above.

### 4a. Simulated run (no requests)

A fake tuev-nord.de with known truth (a reused id, an office at a station's address, a bracket that is a place name, an unclassified name, list-only pages). Must PASS.

In [ ]:
from core import paths
from tuev_nord import run as nord

sim = nord.run(mode="simulated")
print(sim.status, "-", sim.summary["outcome"], "|", sim.summary["counts"])
print(sim.run_dir.relative_to(paths.project_root()))
sim.frames["hu"][["location_id", "name_clean", "site_type", "service_type", "dq_flags"]].head(8)

### 4b. Full run (live, ~12 requests, under a minute)

**Live requests.** Needs `confirm_full_run=True`. The committed default is `False`: with it the cell stops with an error on purpose before any request is sent. Pace: 2-4 s random pause, one request at a time, fresh session without cookies, at most 3 retries with backoff; a block page, 403/503 or a page without the map attribute stops the run (never parsed as "0 stations").

Expect about 497 HU sites in 612 records (2026-10-06), but the figures come from the run: compare them with the summary, not with this text.

In [ ]:
from core import paths
from tuev_nord import run as nord

# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = nord.run(mode="full", confirm_full_run=False)

In [ ]:
# After a full run: outcome, failed checks, counts
print(full.status, "-", full.summary["outcome"], "|", full.summary["counts"])
print("Drop check:", full.summary["drop_check"])
print(full.run_dir.relative_to(paths.project_root()))
pd.DataFrame(full.summary["checks"]).query("not ok")[["severity", "check", "detail"]]

## 5. Compare two runs (change detection)

Which stations appeared or disappeared between two runs of `output/`, by the unique key (id + normalised address). Set `OLD` and `NEW` to run-folder names.

In [ ]:
print("Runs in output/:", [p.name for p in official_runs()])

OLD, NEW = None, None  # e.g. "2026-10-06_1116_rebuild", "2026-11-03_0900_full"
if OLD and NEW:
    def load(name):
        f = next((OUT / name).glob("tuev_nord_hu_*.csv"))
        return pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False).set_index("source_key")
    old, new = load(OLD), load(NEW)
    print(f"{len(old)} -> {len(new)} HU sites")
    print("new:", sorted(set(new.index) - set(old.index)))
    print("gone:", sorted(set(old.index) - set(new.index)))
    both = sorted(set(old.index) & set(new.index))
    cols = ["name_clean", "service_type", "site_type", "operator_name", "street", "zip", "city", "latitude", "longitude"]
    changed = [(k, c, old.at[k, c], new.at[k, c]) for k in both for c in cols if old.at[k, c] != new.at[k, c]]
    display(pd.DataFrame(changed, columns=["source_key", "field", "old", "new"]))